# FDCPA Rule Classifier — Three-Way Evaluation

Compare three models on the held-out test set:
1. **o3-mini** — ceiling baseline (OpenAI API)
2. **Qwen2.5-3B-Instruct (base)** — zero-shot baseline (local)
3. **Qwen2.5-3B-Instruct + QLoRA adapter** — fine-tuned model from HF Hub

**Kaggle setup:**
- GPU enabled (T4), Internet ON
- Add Kaggle Secret: `OPENAI_API_KEY`
- Upload `test.jsonl` as part of `fdcpa-data` dataset

In [1]:
# Step 1: Clone repo and install dependencies
!git clone https://github.com/ree2raz/fdcpa-rule-classifier.git /kaggle/working/fdcpa-rule-classifier
!pip install -q transformers>=4.45.0 peft>=0.13.0 trl>=0.11.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets>=3.0.0 python-dotenv tqdm scikit-learn matplotlib seaborn openai

import sys, os
from pathlib import Path

sys.path.insert(0, '/kaggle/working/fdcpa-rule-classifier/src')
os.chdir('/kaggle/working')

# Set environment variable for CUDA memory fragmentation
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

Cloning into '/kaggle/working/fdcpa-rule-classifier'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 50 (delta 15), reused 36 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 304.06 KiB | 4.54 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [2]:
# Step 2: Set API keys
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ['OPENAI_API_KEY'] = user_secrets.get_secret('OPENAI_API_KEY')

In [3]:
# Step 3: Locate test data
import shutil

repo_data = Path('/kaggle/working/fdcpa-rule-classifier/data')
kaggle_input = Path('/kaggle/input/fdcpa-data')

# Try common Kaggle dataset locations
test_path = None
candidates = [
    kaggle_input / 'test.jsonl',
    Path('/kaggle/input/datasets/ree2raz/fdcpa-data/test.jsonl'),
    repo_data / 'test.jsonl',
]
for c in candidates:
    if c.exists():
        test_path = c
        break

if test_path is None:
    # Search all Kaggle input dirs
    for p in sorted(Path('/kaggle/input').rglob('test.jsonl')):
        test_path = p
        break

if test_path:
    # Copy into repo data dir so module defaults work
    repo_data.mkdir(parents=True, exist_ok=True)
    shutil.copy2(test_path, repo_data / 'test.jsonl')
    print(f'Test data found: {test_path}')
else:
    raise FileNotFoundError('test.jsonl not found in /kaggle/input. Upload it as a Kaggle dataset.')

# Verify
from fdcpa_classifier import load_jsonl
import pandas as pd
test_examples = load_jsonl(repo_data / 'test.jsonl')
df_test = pd.DataFrame(test_examples)
print(f'Test set: {len(test_examples)} examples')
print(f'Verdict balance: {df_test["verdict"].value_counts().to_dict()}')

Test data found: /kaggle/input/datasets/ree2raz/fdcpa-data/test.jsonl
Test set: 39 examples
Verdict balance: {'pass': 23, 'fail': 16}


## 2. Run Fine-tuned Model Inference (from HF Hub)

In [6]:
from fdcpa_classifier.eval import run_inference_finetuned

ft_results = run_inference_finetuned(
    model_path='ree2raz/fdcpa-rule-classifier-qlora',
    base_model='Qwen/Qwen2.5-3B-Instruct',
    test_path='/kaggle/working/fdcpa-rule-classifier/data/test.jsonl',
)
print(f'Fine-tuned: {len(ft_results)} predictions')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]


Fine-tuned inference:   0%|          | 0/39 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Fine-tuned inference: 100%|██████████| 39/39 [04:16<00:00,  6.57s/it]

Fine-tuned: 39 predictions


## 3. Run Base Model Inference

In [7]:
from fdcpa_classifier.eval import run_inference_base

base_results = run_inference_base(
    model_name='Qwen/Qwen2.5-3B-Instruct',
    test_path='/kaggle/working/fdcpa-rule-classifier/data/test.jsonl',
)
print(f'Base model: {len(base_results)} predictions')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


Base model inference: 100%|██████████| 39/39 [02:40<00:00,  4.12s/it]

Base model: 39 predictions


## 4. Run OpenAI Baseline Inference

In [5]:
from fdcpa_classifier.eval import run_inference_openai

openai_results = run_inference_openai(
    test_path='/kaggle/working/fdcpa-rule-classifier/data/test.jsonl',
)
print(f'OpenAI: {len(openai_results)} predictions')

OpenAI o3-mini inference: 100%|██████████| 39/39 [01:21<00:00,  2.10s/it]

OpenAI API: 14216 in + 8204 out tokens
OpenAI: 39 predictions


## 5. Comparison Table

In [8]:
from fdcpa_classifier.eval import compute_metrics

all_results = {
    'o3-mini': openai_results,
    'Qwen Base': base_results,
    'Qwen QLoRA': ft_results,
}

print(f"{'Model':<20} {'Accuracy':>10} {'F1 (macro)':>12} {'Parse Rate':>12} {'Avg Latency':>12}")
print('-' * 68)
for name, results in all_results.items():
    m = compute_metrics(results)
    o = m['overall']
    print(f"{name:<20} {o['accuracy']:>10.3f} {o['f1_macro']:>12.3f} {o['parse_success_rate']:>12.1%} {o['avg_latency_ms']:>10.0f}ms")

Model                  Accuracy   F1 (macro)   Parse Rate  Avg Latency
--------------------------------------------------------------------
o3-mini                   1.000        1.000        46.2%       2096ms
Qwen Base                 0.769        0.769       100.0%       4112ms
Qwen QLoRA                0.846        0.846       100.0%       6568ms


## 6. Save Results and Generate Plots

In [9]:
from fdcpa_classifier.eval import save_results
from pathlib import Path

results_dir = Path('/kaggle/working/fdcpa-rule-classifier/results')
results_dir.mkdir(parents=True, exist_ok=True)

serialized = save_results(all_results, output_path=results_dir / 'eval_results.json')

Results saved to /kaggle/working/fdcpa-rule-classifier/results/eval_results.json


In [10]:
from fdcpa_classifier.metrics import plot_per_rule_f1, plot_confusion_matrices, plot_cost_comparison

plot_per_rule_f1(serialized, output_path=results_dir / 'per_rule_f1.png')
plot_confusion_matrices(serialized, output_path=results_dir / 'confusion_matrices.png')
plot_cost_comparison(serialized, output_path=results_dir / 'cost_comparison.png')

Saved: /kaggle/working/fdcpa-rule-classifier/results/per_rule_f1.png
Saved: /kaggle/working/fdcpa-rule-classifier/results/confusion_matrices.png
Saved: /kaggle/working/fdcpa-rule-classifier/results/cost_comparison.png


## 7. Failure Analysis

In [11]:
print('=' * 80)
print("FAILURE ANALYSIS: Fine-tuned model's worst failures")
print('=' * 80)

failures = []
for i, (ex, pred) in enumerate(zip(test_examples, ft_results)):
    if pred['predicted'] != ex['verdict']:
        failures.append((i, ex, pred))

failures.sort(key=lambda x: len(x[1]['transcript_chunk']), reverse=True)
print(f'Total misclassifications: {len(failures)} / {len(test_examples)}')

for idx, (i, ex, pred) in enumerate(failures[:5]):
    print(f"\n{'_' * 80}")
    print(f'Failure #{idx+1} — Example {i}')
    print(f"Rule: {ex['rule_id']} | Actual: {ex['verdict']} | Predicted: {pred['predicted']}")
    print(f"\nTranscript:\n{ex['transcript_chunk'][:500]}...")
    print(f"\nModel raw output: {pred['raw_output'][:300]}")

FAILURE ANALYSIS: Fine-tuned model's worst failures
Total misclassifications: 6 / 39

________________________________________________________________________________
Failure #1 — Example 18
Rule: FDCPA-010 | Actual: pass | Predicted: fail

Transcript:
Agent: Hello, this is Dana from [AGENCY_NAME] regarding your account number [ACCOUNT_NUMBER]. I wanted to follow up on your balance.
Consumer: I’m disputing this debt. I mailed a dispute letter 10 days ago.
Agent: Thank you for letting me know. We do have a dispute flag on file. I’m reviewing if verification has been completed yet.
Consumer: I haven’t received anything.
Agent: From what I see, we began preparing verification documents, but they have not yet been sent. However, I must inform you ...

Model raw output: {"verdict": "fail", "reasoning": "The agent acknowledges receiving a written dispute but continues sending automated statements and making collection calls, which may be considered collection activity despite the intent to p

## 8. Summary

In [12]:
print('\n' + '=' * 60)
print('KEY FINDINGS')
print('=' * 60)
print("""
1. [Fill in after running eval] Where fine-tuning helps most
2. [Fill in after running eval] Where fine-tuning doesn't help
3. [Fill in after running eval] Cost/latency tradeoff
4. [Fill in after running eval] Composability with Scrutiny
""")


KEY FINDINGS

1. [Fill in after running eval] Where fine-tuning helps most
2. [Fill in after running eval] Where fine-tuning doesn't help
3. [Fill in after running eval] Cost/latency tradeoff
4. [Fill in after running eval] Composability with Scrutiny

